# <font color="#418FDE" size="6.5" uppercase>**Regression vergleichen**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Trainieren lineare, regularisierte und robuste Regressionsmodelle in Pipelines. 
- Vergleichen nichtlineare Regressoren wie k-NN, Bäume, Random Forest, Boosting und SVR. 
- Bewerten Regressionsmodelle mit MAE, RMSE, R², Lernkurven und Laufzeit. 


## **1. Lineare Regressionen**

### **1.1. Mehrere Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_01_01.jpg?v=1784797374" width="250">



>* Zielgröße aus mehreren Merkmalen erklären
>* Gewichte zeigen gemeinsame Einflüsse

>* Pipelines bereiten verschiedene Merkmale einheitlich vor
>* Saubere Trennung verhindert Datenlecks und Überschätzung

>* Gewichte nur im Merkmalskontext deuten
>* Transparentes Basismodell für weitere Regressionen



In [ ]:
#@title Python-Code - Mehrere Merkmale

# Wir trainieren eine lineare Regression mit mehreren Merkmalen.
# Eine Pipeline skaliert Merkmale nur mit Trainingsdaten.
# Die Ausgabe zeigt Fehler und gelernte Gewichte.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Der Diabetes-Datensatz ist klein und bereits lokal verfügbar.
diabetes = load_diabetes(as_frame=True)
features = diabetes.frame[["bmi", "bp", "s5"]]
target = diabetes.target

# Diese Prüfung macht die erwartete Tabellenform sichtbar.
if features.shape[1] != 3:
    raise ValueError("Es werden genau drei Merkmale erwartet.")

# Der Split trennt Training und Test für eine faire Bewertung.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Die Pipeline skaliert zuerst und trainiert danach das Modell.
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

# Vorhersagen auf Testdaten zeigen die Leistung auf neuen Fällen.
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

# Die Koeffizienten gehören zu den skalierten Merkmalen.
linear_model = model.named_steps["linearregression"]
coefficients = pd.Series(linear_model.coef_, index=features.columns)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsfälle: {len(X_train)}, Testfälle: {len(X_test)}")
print(f"MAE auf Testdaten: {mae:.1f} Zielpunkte")
print("Gewichte nach Skalierung:")
print(coefficients.round(1).to_string())



### **1.2. Regularisierung verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_01_02.jpg?v=1784797376" width="250">



>* Regularisierung macht lineare Modelle stabiler
>* Sie begrenzt übergroße Koeffizienten gegen Überanpassung

>* Regularisierung bevorzugt einfache, stabile Erklärungen
>* Ridge dämpft, Lasso wählt, Elastic Net kombiniert

>* Merkmale vor Regularisierung sinnvoll standardisieren
>* Regularisierungsstärke per Validierung passend wählen



In [ ]:
#@title Python-Code - Regularisierung verstehen

# Wir vergleichen lineare Modelle mit Regularisierung.
# Skalierung macht Koeffizienten fair vergleichbar.
# Die Ausgabe zeigt Fehler und Koeffizienten.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine Daten mit korrelierten Merkmalen.
rng = np.random.default_rng(42)
base = rng.normal(size=160)
noise = rng.normal(scale=0.15, size=160)

# Zwei Merkmale enthalten fast dieselbe Information.
feature_one = base
feature_two = base + noise
feature_three = rng.normal(size=160)

# Das Ziel hängt hauptsächlich vom gemeinsamen Signal ab.
X = np.column_stack([feature_one, feature_two, feature_three])
y = 80 * base + rng.normal(scale=18, size=160)

# Diese Prüfung schützt vor unerwarteten Formproblemen.
if X.shape != (160, 3) or y.shape[0] != 160:
    raise ValueError("Die Beispieldaten haben eine unerwartete Form.")

# Der Testteil simuliert neue, unbekannte Fälle.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Beide Modelle nutzen dieselbe faire Skalierungspipeline.
linear_model = make_pipeline(StandardScaler(), LinearRegression())
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=10.0))

# Jetzt lernen beide Modelle nur aus den Trainingsdaten.
linear_model.fit(X_train, y_train)
ridge_model.fit(X_train, y_train)

# Wir bewerten die Vorhersagen auf unbekannten Testdaten.
linear_pred = linear_model.predict(X_test)
ridge_pred = ridge_model.predict(X_test)

# RMSE ist ein Fehlermaß in Zielgrößeneinheiten.
linear_rmse = mean_squared_error(y_test, linear_pred) ** 0.5
ridge_rmse = mean_squared_error(y_test, ridge_pred) ** 0.5

# Ridge verteilt Gewicht oft stabiler auf ähnliche Merkmale.
linear_coef = linear_model.named_steps["linearregression"].coef_
ridge_coef = ridge_model.named_steps["ridge"].coef_

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Lineare Regression RMSE: {linear_rmse:.2f}")
print(f"Ridge Regression RMSE: {ridge_rmse:.2f}")
print("Koeffizienten linear: " + str(np.round(linear_coef, 2).tolist()))
print("Koeffizienten Ridge:  " + str(np.round(ridge_coef, 2).tolist()))



### **1.3. Residuen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_01_03.jpg?v=1784797378" width="250">



>* Residuen zeigen Fehler einzelner Vorhersagen.
>* Muster weisen auf Modellprobleme hin.

>* Residuen bewerten die gesamte Pipeline.
>* Sie zeigen Muster, Ausreißer und Robustheit.

>* Residuen zeigen Fehlerstrukturen jenseits von Kennzahlen
>* Sie decken Verzerrungen und Modellierungsbedarf auf



In [ ]:
#@title Python-Code - Residuen verstehen

# Dieses Beispiel macht Residuen einer Regression sichtbar.
# Wir trainieren eine Pipeline mit linearer Regression.
# Das Diagramm zeigt zufällige oder systematische Fehler.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Kleine synthetische Daten halten das Beispiel übersichtlich.
rng = np.random.default_rng(42)
living_area = rng.uniform(30, 160, size=80)
noise = rng.normal(0, 25000, size=80)

# Der wahre Zusammenhang ist leicht gekrümmt.
price = 50000 + 2800 * living_area + 8 * living_area**2 + noise
X = living_area.reshape(-1, 1)
y = price

# Eine einfache Prüfung verhindert unpassende Formen.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte müssen gleich viele Zeilen haben.")

# Die Pipeline skaliert zuerst und trainiert danach linear.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

# Residuen sind beobachtete Werte minus Vorhersagen.
y_pred = model.predict(X_test)
residuals = y_test - y_pred
mae = mean_absolute_error(y_test, y_pred)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testpunkte: {len(y_test)}")
print(f"MAE: {mae:,.0f} Euro")

# Ein Residuenplot zeigt Muster, die Kennzahlen verbergen können.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(y_pred, residuals, alpha=0.8, label="Testdaten")
ax.axhline(0, color="black", linewidth=1, label="kein Fehler")
ax.set_title("Residuen einer linearen Regressions-Pipeline")
ax.set_xlabel("Vorhergesagter Preis in Euro")
ax.set_ylabel("Residuum in Euro")
ax.legend()
plt.show()



## **2. Nichtlineare Regression**

### **2.1. Polynome und Splines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_02_01.jpg?v=1784797380" width="250">



>* Polynome modellieren gekrümmte Zusammenhänge flexibel
>* Höhere Grade erhöhen Überanpassungsrisiko

>* Splines modellieren glatte lokale Abschnitte
>* Oft stabiler als hochgradige Polynome

>* Polynome und Splines bleiben gut erklärbar
>* Modellflexibilität muss zu Daten passen



In [ ]:
#@title Python-Code - Polynome und Splines

# Wir vergleichen Polynome und Splines anschaulich.
# Beide erzeugen nichtlineare Merkmale für Regression.
# Die Grafik zeigt unterschiedliche Kurvenflexibilität.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import SplineTransformer
from sklearn.metrics import mean_squared_error

# Diese Daten enthalten einen glatten nichtlinearen Zusammenhang.
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 80)
noise = rng.normal(0, 0.25, size=x.shape)
y = np.sin(x) + 0.15 * x + noise

# Scikit-learn erwartet Merkmale als zweidimensionale Tabelle.
X = x.reshape(-1, 1)
if X.shape != (80, 1):
    raise ValueError("Die Beispieldaten haben eine unerwartete Form.")

# Ein Polynom nutzt eine globale gekrümmte Kurve.
polynomial_model = make_pipeline(
    PolynomialFeatures(degree=5, include_bias=False),
    LinearRegression(),
)
polynomial_model.fit(X, y)

# Ein Spline nutzt mehrere glatte lokale Kurvenstücke.
spline_model = make_pipeline(
    SplineTransformer(n_knots=6, degree=3, include_bias=False),
    LinearRegression(),
)
spline_model.fit(X, y)

# Wir bewerten beide Modelle auf denselben Trainingspunkten.
polynomial_rmse = np.sqrt(mean_squared_error(y, polynomial_model.predict(X)))
spline_rmse = np.sqrt(mean_squared_error(y, spline_model.predict(X)))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"RMSE Polynom Grad 5: {polynomial_rmse:.3f}")
print(f"RMSE Spline mit 6 Knoten: {spline_rmse:.3f}")

# Für die Kurven zeichnen wir viele geordnete x-Werte.
x_grid = np.linspace(0, 10, 300)
X_grid = x_grid.reshape(-1, 1)
polynomial_curve = polynomial_model.predict(X_grid)
spline_curve = spline_model.predict(X_grid)

# Die Grafik macht globale und lokale Flexibilität sichtbar.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, s=25, alpha=0.65, label="Beobachtungen")
ax.plot(x_grid, polynomial_curve, label="Polynom Grad 5")
ax.plot(x_grid, spline_curve, label="Spline")

ax.set_title("Polynome und Splines für nichtlineare Regression")
ax.set_xlabel("Merkmal x")
ax.set_ylabel("Zielwert y")
ax.legend()
plt.show()



### **2.2. Robuste Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_02_02.jpg?v=1784797382" width="250">



>* Schätzt Trends trotz Ausreißern und Messfehlern
>* Bleibt stabil bei unvollkommenen Daten

>* Robuste Regression macht Schätzungen widerstandsfähiger.
>* Komplexität und Robustheit getrennt vergleichen.

>* Stabilität zählt neben einzelnen Fehlerkennzahlen
>* Robuste Modelle zeigen einflussreiche Ausreißer



In [ ]:
#@title Python-Code - Robuste Regression

# Wir vergleichen robuste und klassische Regression.
# Ausreißer sollen den Unterschied sichtbar machen.
# Die Grafik zeigt stabilere robuste Vorhersagen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen einfache Daten mit wenigen Ausreißern.
rng = np.random.default_rng(42)
area = rng.uniform(30, 120, size=80)
rent = 8.0 * area + 250.0 + rng.normal(0, 45, size=80)

# Einige Luxusobjekte verzerren die Zielwerte stark nach oben.
outlier_index = np.array([5, 18, 41, 63])
rent[outlier_index] = rent[outlier_index] + 900.0
X = area.reshape(-1, 1)
y = rent

# Eine kleine Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte müssen gleich viele Zeilen haben.")

# Der Split trennt Training und Test fair voneinander.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Beide Modelle nutzen dieselbe Pipeline-Struktur.
linear_model = make_pipeline(StandardScaler(), LinearRegression())
robust_model = make_pipeline(StandardScaler(), HuberRegressor(max_iter=200))

# Die Modelle lernen aus denselben Trainingsdaten.
linear_model.fit(X_train, y_train)
robust_model.fit(X_train, y_train)

# MAE bewertet typische absolute Fehler in Euro.
linear_pred = linear_model.predict(X_test)
robust_pred = robust_model.predict(X_test)
linear_mae = mean_absolute_error(y_test, linear_pred)
robust_mae = mean_absolute_error(y_test, robust_pred)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"MAE Lineare Regression: {linear_mae:.1f} Euro")
print(f"MAE Robuste Regression: {robust_mae:.1f} Euro")

# Eine Linie pro Modell zeigt den Einfluss der Ausreißer.
area_grid = np.linspace(30, 120, 100).reshape(-1, 1)
linear_line = linear_model.predict(area_grid)
robust_line = robust_model.predict(area_grid)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train[:, 0], y_train, alpha=0.65, label="Trainingsdaten")
ax.plot(area_grid[:, 0], linear_line, label="Lineare Regression")
ax.plot(area_grid[:, 0], robust_line, label="Robuste Regression")
ax.set_title("Robuste Regression bei Ausreißern")
ax.set_xlabel("Wohnfläche in Quadratmetern")
ax.set_ylabel("Miete in Euro")
ax.legend()
plt.show()



### **2.3. Nachbarn und Bäume**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_02_03.jpg?v=1784797384" width="250">



>* k-NN nutzt ähnliche Fälle für Vorhersagen
>* Skalierung und Nachbarschaftsgröße beeinflussen stark

>* Bäume modellieren nichtlineare Regeln verständlich
>* Tiefe Bäume brauchen Regularisierung gegen Overfitting

>* Ensembles machen Baum-Regression stabiler und genauer
>* Vergleiche auch Laufzeit, Deutbarkeit und Zuverlässigkeit



In [ ]:
#@title Python-Code - Nachbarn und Bäume

# Wir vergleichen Nachbarn und Bäume bei Regression.
# Nichtlineare Modelle lernen unterschiedliche lokale Muster.
# Die Grafik zeigt Vorhersagen und Testfehler.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Wir erzeugen kleine, gekrümmte Regressionsdaten.
rng = np.random.default_rng(42)
X = np.linspace(0, 10, 120).reshape(-1, 1)
noise = rng.normal(0, 0.35, size=X.shape[0])
y = np.sin(X[:, 0]) + 0.25 * X[:, 0] + noise

# Diese Prüfung schützt vor unerwarteten Formfehlern.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte müssen gleich viele Zeilen haben.")

# Der Testteil bleibt für einen fairen Vergleich ungesehen.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# k-NN braucht Skalierung, der Baum kommt ohne Skalierung aus.
knn_model = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=7))
tree_model = DecisionTreeRegressor(max_depth=4, random_state=42)

# Beide Modelle lernen aus denselben Trainingsdaten.
knn_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

# Wir bewerten beide Modelle auf denselben Testdaten.
knn_mae = mean_absolute_error(y_test, knn_model.predict(X_test))
tree_mae = mean_absolute_error(y_test, tree_model.predict(X_test))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"k-NN MAE auf Testdaten: {knn_mae:.3f}")
print(f"Baum MAE auf Testdaten: {tree_mae:.3f}")

# Ein feines Raster macht die Modellformen sichtbar.
X_grid = np.linspace(0, 10, 300).reshape(-1, 1)
knn_curve = knn_model.predict(X_grid)
tree_curve = tree_model.predict(X_grid)

# Die Kurven zeigen glatte Nachbarschaften und stufige Baumregeln.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train[:, 0], y_train, s=22, alpha=0.55, label="Trainingsdaten")
ax.plot(X_grid[:, 0], knn_curve, linewidth=2, label="k-NN")
ax.plot(X_grid[:, 0], tree_curve, linewidth=2, label="Entscheidungsbaum")

ax.set_title("Nichtlineare Regression mit Nachbarn und Baum")
ax.set_xlabel("Merkmal x")
ax.set_ylabel("Zielwert y")
ax.legend()
plt.show()



## **3. Regressionsmodelle vergleichen**

### **3.1. Boosting und SVR**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_03_01.jpg?v=1784797386" width="250">



>* Boosting und SVR modellieren komplexe Zusammenhänge
>* MAE, RMSE und R² gemeinsam bewerten

>* RMSE, MAE und R² gemeinsam deuten
>* SVR skalieren, Boosting gegen Überanpassung regulieren

>* Lernkurven zeigen Nutzen, Grenzen und Überanpassung
>* Laufzeit und Zuverlässigkeit fair mitbewerten



In [ ]:
#@title Python-Code - Boosting und SVR

# Wir vergleichen Boosting und SVR fair.
# Metriken zeigen unterschiedliche Fehlerperspektiven.
# Die Grafik zeigt Genauigkeit und Laufzeit.

from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

# Ein kleiner Datensatz enthält absichtlich nichtlineare Muster.
rng = np.random.default_rng(42)
X, y_linear = make_regression(
    n_samples=700, n_features=6, noise=18, random_state=42
)

# Nichtlinearität macht einfache Muster realistischer.
y = y_linear + 35 * np.sin(X[:, 0]) + 20 * X[:, 1] * X[:, 2]

# Diese Prüfung macht die Beispielannahmen sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der Testdatensatz bleibt bis zur Bewertung ungesehen.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Boosting lernt viele kleine Korrekturen nacheinander.
boosting_model = GradientBoostingRegressor(
    n_estimators=80, learning_rate=0.08, max_depth=2, random_state=42
)

# SVR braucht Skalierung, deshalb nutzen wir eine Pipeline.
svr_model = make_pipeline(
    StandardScaler(), SVR(kernel="rbf", C=20, epsilon=8, gamma="scale")
)

# Wir messen Trainingsdauer ohne zusätzliche Bibliotheken.
models = [("Boosting", boosting_model), ("SVR", svr_model)]
results = []

for name, model in models:
    start_time = datetime.now()
    model.fit(X_train, y_train)
    elapsed = datetime.now() - start_time
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    results.append((name, mae, rmse, r2, elapsed.total_seconds()))

# Drei Kennzahlen und Laufzeit ergeben ein ausgewogeneres Bild.
print(f"scikit-learn Version: {sklearn.__version__}")
for name, mae, rmse, r2, seconds in results:
    print(f"{name}: MAE={mae:.1f}, RMSE={rmse:.1f}, R²={r2:.3f}, Zeit={seconds:.3f}s")

# Die Balken vergleichen RMSE als fehlerorientierte Hauptmetrik.
model_names = [row[0] for row in results]
rmse_values = [row[2] for row in results]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(model_names, rmse_values, color=["#4C78A8", "#F58518"])
ax.set_title("Boosting und SVR: RMSE auf Testdaten")
ax.set_xlabel("Modell")
ax.set_ylabel("RMSE in Zieleinheiten")
plt.show()



### **3.2. Lernkurven verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_03_02.jpg?v=1784797388" width="250">



>* Trainings- und Validierungsleistung über Datenmengen vergleichen
>* Erkennen, ob mehr Daten oder Modelländerungen helfen

>* Große Leistungslücke zeigt oft Überanpassung.
>* Hohe ähnliche Fehler deuten auf Unteranpassung.

>* Lernkurven mit Metriken und Laufzeit vergleichen
>* Sinkende Fehler gegen Rechenaufwand abwägen



In [ ]:
#@title Python-Code - Lernkurven verstehen

# Diese Lernkurve zeigt Training und Validierung.
# Wir erkennen Unteranpassung und Überanpassung visuell.
# Die Grafik hilft beim Modellvergleich.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.model_selection import learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import make_scorer
from sklearn.metrics import mean_absolute_error

# Ein kleiner Datensatz macht die Lernkurve schnell berechenbar.
features, target = make_regression(
    n_samples=220,
    n_features=1,
    noise=18,
    random_state=42,
)

# Eine leichte Nichtlinearität macht das Beispiel realistischer.
target = target + 35 * np.sin(features[:, 0])

# Diese Prüfung verhindert unklare Fehler bei falschen Datenformen.
if features.shape[0] != target.shape[0]:
    raise ValueError("Merkmale und Zielwerte müssen gleich viele Zeilen haben.")

# Eine Pipeline verhindert Datenleckage bei der Kreuzvalidierung.
model = make_pipeline(
    PolynomialFeatures(degree=4, include_bias=False),
    Ridge(alpha=1.0),
)

# Negative MAE-Werte werden später in positive Fehler umgerechnet.
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

# learning_curve trainiert dasselbe Modell mit wachsenden Datenmengen.
train_sizes, train_scores, validation_scores = learning_curve(
    model, features, target, train_sizes=np.linspace(0.15, 1.0, 6),
    cv=5, scoring=mae_scorer
)

# Mittelwerte über die Folds machen die Kurven stabiler.
train_mae = -train_scores.mean(axis=1)
validation_mae = -validation_scores.mean(axis=1)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Kleinste Trainingsmenge: {train_sizes[0]} Beispiele")
print(f"Validierungs-MAE am Ende: {validation_mae[-1]:.1f}")

# Die Kurven zeigen, ob mehr Daten wahrscheinlich helfen.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_sizes, train_mae, marker="o", label="Training MAE")
ax.plot(train_sizes, validation_mae, marker="o", label="Validierung MAE")

ax.set_title("Lernkurve eines Regressionsmodells")
ax.set_xlabel("Anzahl Trainingsbeispiele")
ax.set_ylabel("MAE, kleiner ist besser")
ax.legend()

plt.show()



### **3.3. Modelle fair vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_10/Lecture_A/image_03_03.jpg?v=1784797390" width="250">



>* Gleiche Bedingungen für alle Modelle schaffen
>* Datenlecks durch saubere Trennung vermeiden

>* Mehrere Metriken gemeinsam bewerten
>* Lernkurven, Stabilität und Praxisnutzen beachten

>* Laufzeit und Einsatzkontext mitbewerten
>* Vergleiche wiederholbar und strukturiert durchführen



In [ ]:
#@title Python-Code - Modelle fair vergleichen

# Wir vergleichen Modelle unter gleichen Bedingungen.
# Kreuzvalidierung bewertet Fehler stabiler als Zufall.
# Die Tabelle zeigt faire Regressionsmetriken.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_diabetes

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen Regressionsdatensatz aus scikit-learn.
data = load_diabetes()
X = data.data
y = data.target

# Eine einfache Prüfung verhindert unklare Datenprobleme.
if X.shape[0] != y.shape[0] or X.shape[0] < 50:
    raise ValueError("Die Daten passen nicht zur Regressionsaufgabe.")

# Die Pipeline skaliert nur innerhalb jeder Trainingsfaltung.
model = make_pipeline(StandardScaler(), LinearRegression())

# Drei Metriken bewerten unterschiedliche Qualitätsaspekte.
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

# Fünf Faltungen geben jedem Datenpunkt einmal eine Testrolle.
results = cross_validate(
    model,
    X,
    y,
    cv=5,
    scoring=scoring,
    return_train_score=False,
)

# Negative Fehlerwerte werden für die Anzeige positiv gemacht.
summary = pd.DataFrame(
    {
        "Metrik": ["MAE", "RMSE", "R²"],
        "Mittelwert": [
            -np.mean(results["test_MAE"]),
            -np.mean(results["test_RMSE"]),
            np.mean(results["test_R2"]),
        ],
        "Streuung": [
            np.std(-results["test_MAE"]),
            np.std(-results["test_RMSE"]),
            np.std(results["test_R2"]),
        ],
    }
)

# Die Ausgabe bleibt kurz und direkt vergleichbar.
print(f"scikit-learn Version: {sklearn.__version__}")
print("Fairer Vergleich: gleiche Daten, gleiche Faltungen, gleiche Pipeline.")
print(summary.round(2).to_string(index=False))



# <font color="#418FDE" size="6.5" uppercase>**Regression vergleichen**</font>


In this lecture, you learned to:
- Trainieren lineare, regularisierte und robuste Regressionsmodelle in Pipelines. 
- Vergleichen nichtlineare Regressoren wie k-NN, Bäume, Random Forest, Boosting und SVR. 
- Bewerten Regressionsmodelle mit MAE, RMSE, R², Lernkurven und Laufzeit. 

In the next Lecture (Lecture B), we will go over 'Klassifikation vergleichen'